# Clustering TEC21 vs Pre-TEC21 — Nuevas Features v4

Análisis independiente de K-Means para **Pre-TEC21** y **TEC21** con el conjunto de features actualizado.

### Features utilizadas
| Feature | Descripción |
|---------|-------------|
| `PNA` | Promedio Normalizado de Admisión |
| `is_foreign` | Extranjero (1) / Nacional (0) — combinado de `foreign_Yes: Foreigner` |
| `estuvo.prepa_tec` | Egresado de Preparatoria Tec |
| `first.generation.yes` | Primera generación universitaria |
| `has_extracurriculars` | Tiene actividades extracurriculares |
| `parents_exatec_enc` | Padres son EXATEC |
| `total.scholarship.loan` | Beca o préstamo (proporción) |
| `FTE` | Carga académica (Full-Time Equivalent) |

**Variable objetivo:** `retention` (0 = desertor, 1 = retenido)


## 0. Imports y configuración

In [88]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine as cos_dist
from scipy.optimize import linear_sum_assignment

SEED   = 42
K      = 4
DATA   = '../data/dataset_k_means.csv'
OUT    = '../data/'
palette = ['#22c55e', '#3b82f6', '#f59e0b', '#ef4444']

FEATURES = [
    'PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes',
    'has_extracurriculars', 'parents_exatec_enc',
    'total.scholarship.loan', 'FTE'
]

LABELS = {
    'PNA':                   'PNA',
    'is_foreign':            'Extranjero',
    'estuvo.prepa_tec':      'Prepa Tec',
    'first.generation.yes':  'Primera gen.',
    'has_extracurriculars':  'Extracurricular',
    'parents_exatec_enc':    'Padres EXATEC',
    'total.scholarship.loan':'Beca/Préstamo',
    'FTE':                   'Carga (FTE)',
}
print("Configuración lista ✓")


Configuración lista ✓


## 1. Carga y feature engineering

In [89]:
df_raw = pd.read_csv(DATA)

# ── Feature engineering ──────────────────────────────────────────────────
# is_foreign: 1 si foreign_Yes: Foreigner = 1, de lo contrario 0
df_raw['is_foreign'] = df_raw['foreign_Yes: Foreigner'].astype(int)

df_raw['dropout'] = 1 - df_raw['retention']

print(f"Dataset total: n={len(df_raw):,}")
print(f"  Retenidos:   n={df_raw['retention'].sum():,}  ({df_raw['retention'].mean()*100:.2f}%)")
print(f"  Desertores:  n={(df_raw['retention']==0).sum():,}  ({df_raw['dropout'].mean()*100:.2f}%)")
print()
print(f"  Extranjeros: n={df_raw['is_foreign'].sum():,}  ({df_raw['is_foreign'].mean()*100:.2f}%)")
print(f"  Prepa Tec:   n={df_raw['estuvo.prepa_tec'].sum():,}")
print(f"  Primera gen: n={df_raw['first.generation.yes'].sum():,}")


Dataset total: n=77,517
  Retenidos:   n=70,704  (91.21%)
  Desertores:  n=6,813  (8.79%)

  Extranjeros: n=2,638  (3.40%)
  Prepa Tec:   n=36,943
  Primera gen: n=5,773


## 2. Separación Pre-TEC21 / TEC21

In [90]:
pre_df = df_raw[df_raw['educational.model'] == 0].copy().reset_index(drop=True)
tec_df = df_raw[df_raw['educational.model'] == 1].copy().reset_index(drop=True)

print(f"Pre-TEC21: n={len(pre_df):,}  dropout={pre_df['dropout'].mean()*100:.2f}%")
print(f"TEC21:     n={len(tec_df):,}  dropout={tec_df['dropout'].mean()*100:.2f}%")


Pre-TEC21: n=53,010  dropout=8.84%
TEC21:     n=24,507  dropout=8.68%


## 3. Preprocesamiento

In [91]:
def preprocess(subset, feats=None):
    """Normaliza (z-score) las features; elimina columnas con var=0 en el subconjunto."""
    feats = feats or FEATURES
    valid = [f for f in feats if f in subset.columns and subset[f].std() > 0]
    X     = subset[valid].fillna(subset[valid].median()).values.astype(float)
    return StandardScaler().fit_transform(X), valid

def reorder_by_dropout(labels, y):
    order   = np.argsort([y[labels==k].mean() for k in range(K)])
    mapping = {old: new for new, old in enumerate(order)}
    return np.array([mapping[l] for l in labels])

def build_profile(subset, labels, valid_feats, y_out):
    """Z-score de medias por cluster + tasa de deserción y tamaño."""
    df_c  = subset[valid_feats].copy()
    means = df_c.groupby(labels).mean()
    z     = (means - df_c.mean()) / df_c.std()
    z.index.name = 'cluster'
    z['Deserción (%)'] = [y_out[labels==k].mean()*100 for k in range(K)]
    z['n']             = [int((labels==k).sum()) for k in range(K)]
    return z

def plot_heatmap(ax_h, ax_b, profile, title, pal, y_global):
    fcols  = [c for c in profile.columns if c not in ['Deserción (%)', 'n', 'grupo']]
    data   = profile[fcols].values.astype(float)
    pretty = [LABELS.get(f,f) for f in fcols]

    im = ax_h.imshow(data, cmap='RdYlGn_r', aspect='auto', vmin=-2, vmax=2)
    ax_h.set_xticks(range(len(fcols))); ax_h.set_xticklabels(pretty, rotation=40, ha='right', fontsize=9)
    ax_h.set_yticks(range(K)); ax_h.set_yticklabels([f'C{k}' for k in range(K)], fontsize=11, fontweight='bold')
    ax_h.set_title(title, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=ax_h, label='z-score', fraction=0.03, pad=0.02)
    for i in range(K):
        for j in range(len(fcols)):
            v = data[i,j]
            ax_h.text(j, i, f'{v:.2f}', ha='center', va='center',
                      fontsize=8, color='white' if abs(v)>1.2 else 'black')

    vals = profile['Deserción (%)'].values
    ns   = profile['n'].values
    bars = ax_b.bar(range(K), vals, color=pal, edgecolor='white', lw=1.5)
    ax_b.axhline(y_global*100, color='black', ls='--', lw=1.3, label=f'Global {y_global*100:.1f}%')
    ax_b.set_xticks(range(K)); ax_b.set_xticklabels([f'C{k}\n(n={ns[k]:,})' for k in range(K)], fontsize=9)
    ax_b.set_ylabel('Deserción (%)'); ax_b.set_title('Tasa de deserción')
    ax_b.legend(fontsize=9)
    for bar, val in zip(bars, vals):
        ax_b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.25,
                  f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

print("Funciones auxiliares definidas ✓")


Funciones auxiliares definidas ✓


In [92]:
X_pre, feat_pre = preprocess(pre_df)
X_tec, feat_tec = preprocess(tec_df)
y_pre = pre_df['dropout'].values
y_tec = tec_df['dropout'].values

print(f"Features Pre-TEC21 ({len(feat_pre)}): {feat_pre}")
print(f"Features TEC21     ({len(feat_tec)}): {feat_tec}")
print(f"X_pre: {X_pre.shape}  X_tec: {X_tec.shape}")


Features Pre-TEC21 (8): ['PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'total.scholarship.loan', 'FTE']
Features TEC21     (8): ['PNA', 'is_foreign', 'estuvo.prepa_tec', 'first.generation.yes', 'has_extracurriculars', 'parents_exatec_enc', 'total.scholarship.loan', 'FTE']
X_pre: (53010, 8)  X_tec: (24507, 8)


## 4. Selección de K (inercia, Silhouette, Davies-Bouldin)

In [93]:
K_RANGE = range(2, 9)

def compute_k_metrics(X, n=5000):
    idx = np.random.choice(len(X), min(n, len(X)), replace=False)
    Xs  = X[idx]
    iner, sils, dbs = [], [], []
    for k in K_RANGE:
        km  = KMeans(n_clusters=k, random_state=SEED, n_init=15).fit(X)
        iner.append(km.inertia_)
        ls  = km.predict(Xs)
        sils.append(silhouette_score(Xs, ls))
        dbs.append(davies_bouldin_score(Xs, ls))
    return iner, sils, dbs


In [94]:
print("Calculando métricas Pre-TEC21...")
i_pre, s_pre, d_pre = compute_k_metrics(X_pre)
print("Calculando métricas TEC21...")
i_tec, s_tec, d_tec = compute_k_metrics(X_tec)

ks = list(K_RANGE)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Selección de K — Pre-TEC21 y TEC21\n(línea roja = K=4 elegido)',
             fontweight='bold', fontsize=13)
for row, (iner, sil, db, lbl) in enumerate([
    (i_pre,s_pre,d_pre,'Pre-TEC21'),(i_tec,s_tec,d_tec,'TEC21')]):
    axes[row,0].plot(ks,iner,'o-',color='#3b82f6',lw=2); axes[row,0].axvline(K,color='red',ls='--',alpha=.7,label=f'K={K}')
    axes[row,0].set_title(f'{lbl} — Inercia'); axes[row,0].set_xlabel('K'); axes[row,0].legend()
    axes[row,1].plot(ks,sil,'o-',color='#22c55e',lw=2); axes[row,1].axvline(K,color='red',ls='--',alpha=.7)
    axes[row,1].set_title(f'{lbl} — Silhouette'); axes[row,1].set_xlabel('K')
    axes[row,2].plot(ks,db,'o-',color='#ef4444',lw=2); axes[row,2].axvline(K,color='red',ls='--',alpha=.7)
    axes[row,2].set_title(f'{lbl} — Davies-Bouldin'); axes[row,2].set_xlabel('K')
plt.tight_layout()
plt.show()


Calculando métricas Pre-TEC21...
Calculando métricas TEC21...


## 5. K-Means K=4 y métricas

In [95]:
km_pre = KMeans(n_clusters=K, random_state=SEED, n_init=20).fit(X_pre)
km_tec = KMeans(n_clusters=K, random_state=SEED, n_init=20).fit(X_tec)
lbl_pre = reorder_by_dropout(km_pre.labels_, y_pre)
lbl_tec = reorder_by_dropout(km_tec.labels_, y_tec)

sil_pre = silhouette_score(X_pre[:5000], lbl_pre[:5000])
sil_tec = silhouette_score(X_tec[:5000], lbl_tec[:5000])
db_pre  = davies_bouldin_score(X_pre, lbl_pre)
db_tec  = davies_bouldin_score(X_tec, lbl_tec)

print(f"Métricas K=4:")
print(f"  Pre-TEC21  →  Silhouette={sil_pre:.3f}  Davies-Bouldin={db_pre:.3f}")
print(f"  TEC21      →  Silhouette={sil_tec:.3f}  Davies-Bouldin={db_tec:.3f}")
print()
for k in range(K):
    np_ = (lbl_pre==k).sum(); dp = y_pre[lbl_pre==k].mean()*100
    nt_ = (lbl_tec==k).sum(); dt = y_tec[lbl_tec==k].mean()*100
    print(f"C{k}: Pre n={np_:,}({np_/len(pre_df)*100:.1f}%) drop={dp:.1f}% | TEC n={nt_:,}({nt_/len(tec_df)*100:.1f}%) drop={dt:.1f}%")


Métricas K=4:
  Pre-TEC21  →  Silhouette=0.245  Davies-Bouldin=1.236
  TEC21      →  Silhouette=0.241  Davies-Bouldin=1.499

C0: Pre n=21,262(40.1%) drop=5.8% | TEC n=9,248(37.7%) drop=4.7%
C1: Pre n=29,680(56.0%) drop=10.6% | TEC n=11,293(46.1%) drop=10.7%
C2: Pre n=1,667(3.1%) drop=14.2% | TEC n=943(3.8%) drop=10.8%
C3: Pre n=401(0.8%) drop=20.2% | TEC n=3,023(12.3%) drop=12.8%


## 6. Perfiles (z-scores) — Heatmap

In [96]:
prof_pre = build_profile(pre_df, lbl_pre, feat_pre, y_pre)
prof_tec = build_profile(tec_df, lbl_tec, feat_tec, y_tec)

print("Perfiles Pre-TEC21 (z-scores):")
print(prof_pre.round(3).to_string())
print("\nPerfiles TEC21 (z-scores):")
print(prof_tec.round(3).to_string())


Perfiles Pre-TEC21 (z-scores):
           PNA  is_foreign  estuvo.prepa_tec  first.generation.yes  has_extracurriculars  parents_exatec_enc  total.scholarship.loan    FTE  Deserción (%)      n
cluster                                                                                                                                                          
0        0.681      -0.182            -0.010                -0.003                 0.087              -0.016                   0.993  0.077          5.771  21262
1       -0.492      -0.182             0.062                -0.001                 0.087               0.004                  -0.686 -0.055         10.580  29680
2        0.098       5.502            -0.952                -0.025                 0.087               0.035                  -0.377  0.099         14.217   1667
3       -0.090       0.215            -0.087                 0.332               -11.454               0.441                  -0.353 -0.438         20.200    4

In [97]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Perfiles de Clusters — z-scores y Deserción\n(features: PNA, extranjero, Prepa Tec, primera gen., extracurr., padres EXATEC, beca, FTE)',
             fontsize=12, fontweight='bold')
plot_heatmap(axes[0,0], axes[0,1], prof_pre, 'Pre-TEC21', palette, y_pre.mean())
plot_heatmap(axes[1,0], axes[1,1], prof_tec, 'TEC21',     palette, y_tec.mean())
plt.tight_layout()
plt.show()


## 7. Silhouette Plot

In [98]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Silhouette Plot — K=4', fontsize=13, fontweight='bold')
for ax, X, lbl, title in [(axes[0],X_pre,lbl_pre,'Pre-TEC21'),(axes[1],X_tec,lbl_tec,'TEC21')]:
    idx = np.random.choice(len(X), min(4000,len(X)), replace=False)
    sv  = silhouette_samples(X[idx], lbl[idx])
    yl  = 10
    for k in range(K):
        ks  = np.sort(sv[lbl[idx]==k])
        yu  = yl + ks.shape[0]
        ax.fill_betweenx(np.arange(yl,yu), 0, ks, facecolor=palette[k], alpha=.7, label=f'C{k}')
        yl  = yu + 10
    ax.axvline(sv.mean(),color='red',ls='--',lw=1.5,label=f'Media={sv.mean():.3f}')
    ax.set_title(title,fontweight='bold'); ax.set_xlabel('Silhouette'); ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 8. PCA 2D

In [99]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Proyección PCA 2D — K=4 clusters', fontsize=13, fontweight='bold')
for ax, X, lbl, title in [(axes[0],X_pre,lbl_pre,'Pre-TEC21'),(axes[1],X_tec,lbl_tec,'TEC21')]:
    idx = np.random.choice(len(X), min(5000,len(X)), replace=False)
    pca = PCA(n_components=2, random_state=SEED)
    Xp  = pca.fit_transform(X[idx])
    for k in range(K):
        m = lbl[idx]==k
        ax.scatter(Xp[m,0],Xp[m,1],c=palette[k],alpha=.35,s=8,label=f'C{k}')
    ax.set_title(title,fontweight='bold')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.legend(markerscale=4, fontsize=9)
plt.tight_layout()
plt.show()


## 9. Valores reales por cluster

In [100]:
disc = [
    ('PNA',                   'PNA',              80, 100),
    ('estuvo.prepa_tec',      'Prepa Tec',         0,   1),
    ('parents_exatec_enc',    'Padres EXATEC',     0,   1),
    ('is_foreign',            'Extranjero',        0,   1),
    ('first.generation.yes',  'Primera gen.',      0,   1),
    ('has_extracurriculars',  'Extracurricular',   0,   1),
    ('total.scholarship.loan','Beca/Préstamo',     0,   1),
    ('FTE',                   'Carga FTE',       0.5, 1.5),
]
fig, axes = plt.subplots(2, len(disc), figsize=(24, 8))
fig.suptitle('Valores Reales por Cluster — Pre-TEC21 (arriba) · TEC21 (abajo)',
             fontsize=13, fontweight='bold')
for ci, (feat, label, ymin, ymax) in enumerate(disc):
    for ri, (subset, labels, gname) in enumerate([
        (pre_df, lbl_pre,'Pre-TEC21'),(tec_df, lbl_tec,'TEC21')]):
        ax   = axes[ri,ci]
        vals = [subset[feat].values[labels==k].mean() for k in range(K)]
        bars = ax.bar(range(K), vals, color=palette, edgecolor='white', lw=1)
        rng  = ymax - ymin
        ax.set_ylim(ymin-.05*rng, ymax+.2*rng)
        ax.set_xticks(range(K)); ax.set_xticklabels([f'C{k}' for k in range(K)], fontsize=8)
        if ci==0: ax.set_ylabel(gname, fontsize=10, fontweight='bold')
        if ri==0: ax.set_title(label, fontsize=9, fontweight='bold')
        for bar,val in zip(bars,vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.01*rng,
                   f'{val:.2f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()


## 10. Interpretación de perfiles

> Los clusters se ordenan de C0 (menor deserción) a C3 (mayor deserción) dentro de cada modelo educativo.


## 11. Matching Pre-TEC21 ↔ TEC21

**Metodología:** similitud coseno entre centroides z-score usando las features compartidas (no constantes en ambos grupos) + algoritmo húngaro para el emparejamiento 1-a-1 óptimo.


In [101]:
# Estandarizar ambos grupos sobre features compartidas para el matching
feat_shared = [f for f in FEATURES if feat_pre and f in feat_pre
               and feat_tec and f in feat_tec]

X_pre_sh = StandardScaler().fit_transform(
    pre_df[feat_shared].fillna(pre_df[feat_shared].median()))
X_tec_sh = StandardScaler().fit_transform(
    tec_df[feat_shared].fillna(tec_df[feat_shared].median()))

c_pre = np.array([X_pre_sh[lbl_pre==k].mean(axis=0) for k in range(K)])
c_tec = np.array([X_tec_sh[lbl_tec==k].mean(axis=0) for k in range(K)])

sim_matrix = np.zeros((K, K))
for i in range(K):
    for j in range(K):
        sim_matrix[i, j] = max(0.0, 1 - cos_dist(c_pre[i], c_tec[j]))

sim_df = pd.DataFrame(sim_matrix,
    index=[f'Pre-C{k}' for k in range(K)],
    columns=[f'TEC-C{k}' for k in range(K)])
print("Matriz de similitud coseno (features compartidas):")
print(sim_df.round(3).to_string())

row_ind, col_ind = linear_sum_assignment(-sim_matrix)
match_pairs = [(r, c, sim_matrix[r,c]) for r,c in zip(row_ind, col_ind)]

print("\nMatching óptimo (algoritmo húngaro):")
for r, c, s in match_pairs:
    dr = y_pre[lbl_pre==r].mean()*100
    dc = y_tec[lbl_tec==c].mean()*100
    tag = '🟢 Sólido' if s>=0.4 else ('🟡 Moderado' if s>=0.15 else '🔴 Débil')
    print(f"  {tag}: Pre-C{r} ({dr:.1f}%) ↔ TEC-C{c} ({dc:.1f}%)  sim={s:.3f}")


Matriz de similitud coseno (features compartidas):
        TEC-C0  TEC-C1  TEC-C2  TEC-C3
Pre-C0   0.946   0.000   0.000   0.002
Pre-C1   0.000   0.884   0.000   0.022
Pre-C2   0.000   0.000   0.999   0.000
Pre-C3   0.000   0.184   0.045   0.000

Matching óptimo (algoritmo húngaro):
  🟢 Sólido: Pre-C0 (5.8%) ↔ TEC-C0 (4.7%)  sim=0.946
  🟢 Sólido: Pre-C1 (10.6%) ↔ TEC-C1 (10.7%)  sim=0.884
  🟢 Sólido: Pre-C2 (14.2%) ↔ TEC-C2 (10.8%)  sim=0.999
  🔴 Débil: Pre-C3 (20.2%) ↔ TEC-C3 (12.8%)  sim=0.000


In [102]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap de similitud
ax_h = axes[0]
im = ax_h.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=max(sim_matrix.max()*1.1, 0.4))
ax_h.set_xticks(range(K)); ax_h.set_xticklabels([f'TEC-C{k}' for k in range(K)], fontsize=11)
ax_h.set_yticks(range(K)); ax_h.set_yticklabels([f'Pre-C{k}' for k in range(K)], fontsize=11)
ax_h.set_title('Similitud Coseno\nPre-TEC21 ↔ TEC21', fontweight='bold', fontsize=12)
plt.colorbar(im, ax=ax_h, label='Similitud coseno', fraction=0.04)
for i in range(K):
    for j in range(K):
        matched = any(r==i and c==j for r,c,_ in match_pairs)
        ax_h.text(j, i, f'{sim_matrix[i,j]:.3f}', ha='center', va='center',
                 fontsize=12, fontweight='bold' if matched else 'normal')
        if matched:
            ax_h.add_patch(plt.Rectangle((j-.49,i-.49), .98, .98,
                           fill=False, edgecolor='red', lw=3))

# Diagrama de flujo
ax_f = axes[1]
ax_f.set_xlim(0,10); ax_f.set_ylim(-.6, K-.4); ax_f.axis('off')
ax_f.set_title('Matching Óptimo — Algoritmo Húngaro', fontweight='bold', fontsize=12)
yp = list(range(K-1,-1,-1))
for k in range(K):
    col = palette[k]
    n_p  = int((lbl_pre==k).sum()); dr_p = y_pre[lbl_pre==k].mean()*100
    ax_f.add_patch(mpatches.FancyBboxPatch((.05,yp[k]-.3),3.2,.6,
        boxstyle='round,pad=0.05',facecolor=col+'33',edgecolor=col,lw=2))
    ax_f.text(1.65,yp[k]+.06,f'Pre-C{k}',fontsize=11,fontweight='bold',color=col,ha='center')
    ax_f.text(1.65,yp[k]-.19,f'n={n_p:,} · {dr_p:.1f}% drop',fontsize=7.5,color='#444',ha='center')
for k in range(K):
    col = palette[k]
    n_t  = int((lbl_tec==k).sum()); dr_t = y_tec[lbl_tec==k].mean()*100
    ax_f.add_patch(mpatches.FancyBboxPatch((6.75,yp[k]-.3),3.2,.6,
        boxstyle='round,pad=0.05',facecolor=col+'33',edgecolor=col,lw=2))
    ax_f.text(8.35,yp[k]+.06,f'TEC-C{k}',fontsize=11,fontweight='bold',color=col,ha='center')
    ax_f.text(8.35,yp[k]-.19,f'n={n_t:,} · {dr_t:.1f}% drop',fontsize=7.5,color='#444',ha='center')
for pi,(r,c,s) in enumerate(match_pairs):
    col = '#16a34a' if s>=.4 else ('#d97706' if s>=.15 else '#dc2626')
    ax_f.annotate('',xy=(6.7,yp[c]),xytext=(3.3,yp[r]),
        arrowprops=dict(arrowstyle='->',color=col,lw=1.5+4*s))
    ax_f.text(5.0,(yp[r]+yp[c])/2+(pi-1.5)*.07,f'sim={s:.3f}',
        fontsize=9,ha='center',color=col,fontweight='bold',
        bbox=dict(facecolor='white',edgecolor=col,pad=1.5,alpha=.85))

plt.tight_layout()
plt.show()


In [103]:
# Comparativo dropout por par
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Deserción: Pre-TEC21 vs TEC21 — Por Cluster y Por Par Emparejado',
             fontsize=13, fontweight='bold')
x, w = np.arange(K), 0.35
ax  = axes[0]
b1  = ax.bar(x-w/2, prof_pre['Deserción (%)'], w, label='Pre-TEC21', color='#3b82f6', alpha=.85)
b2  = ax.bar(x+w/2, prof_tec['Deserción (%)'], w, label='TEC21',     color='#f59e0b', alpha=.85)
ax.set_xticks(x); ax.set_xticklabels([f'C{k}' for k in range(K)])
ax.set_ylabel('Deserción (%)'); ax.set_title('Por cluster'); ax.legend()
for bars in [b1,b2]:
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.2,
               f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
ax2 = axes[1]
plabels = [f'Par {i+1}\nPre-C{r}↔TEC-C{c}\nsim={s:.2f}' for i,(r,c,s) in enumerate(match_pairs)]
dp_p = [y_pre[lbl_pre==r].mean()*100 for r,c,_ in match_pairs]
dt_p = [y_tec[lbl_tec==c].mean()*100 for r,c,_ in match_pairs]
x2   = np.arange(len(match_pairs))
b3   = ax2.bar(x2-w/2, dp_p, w, label='Pre-TEC21', color='#3b82f6', alpha=.85)
b4   = ax2.bar(x2+w/2, dt_p, w, label='TEC21',     color='#f59e0b', alpha=.85)
ax2.set_xticks(x2); ax2.set_xticklabels(plabels, fontsize=8)
ax2.set_ylabel('Deserción (%)'); ax2.set_title('Por par emparejado'); ax2.legend()
for bars in [b3,b4]:
    for bar in bars:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.2,
               f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()


## 12. Exportar resultados

In [104]:
prof_pre['grupo'] = 'Pre-TEC21'
prof_tec['grupo'] = 'TEC21'
pd.concat([prof_pre, prof_tec]).to_csv(OUT+'v4_perfiles.csv')

match_out = []
for r,c,s in match_pairs:
    match_out.append({'par': f'Pre-C{r}↔TEC-C{c}', 'pre_cluster':r, 'tec_cluster':c,
        'similitud': round(s,4),
        'dropout_pre_%': round(y_pre[lbl_pre==r].mean()*100,2),
        'dropout_tec_%': round(y_tec[lbl_tec==c].mean()*100,2),
        'n_pre': int((lbl_pre==r).sum()), 'n_tec': int((lbl_tec==c).sum()),
        'calidad': 'sólido' if s>=.4 else ('moderado' if s>=.15 else 'débil')})
pd.DataFrame(match_out).to_csv(OUT+'v4_matching.csv', index=False)
print("✓ v4_perfiles.csv y v4_matching.csv guardados")


✓ v4_perfiles.csv y v4_matching.csv guardados


## A. Resumen ejecutivo de métricas

Tabla consolidada con las métricas de calidad del clustering, tamaño y deserción de cada cluster.


In [105]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# Recopilar métricas completas
sil_full_pre = silhouette_score(X_pre, lbl_pre)
sil_full_tec = silhouette_score(X_tec, lbl_tec)
db_full_pre  = davies_bouldin_score(X_pre, lbl_pre)
db_full_tec  = davies_bouldin_score(X_tec, lbl_tec)
iner_pre = KMeans(n_clusters=K, random_state=SEED, n_init=20).fit(X_pre).inertia_
iner_tec = KMeans(n_clusters=K, random_state=SEED, n_init=20).fit(X_tec).inertia_

# Tabla global
global_metrics = pd.DataFrame({
    'Grupo':       ['Pre-TEC21', 'TEC21'],
    'n total':     [len(pre_df), len(tec_df)],
    'Dropout global (%)': [y_pre.mean()*100, y_tec.mean()*100],
    'Silhouette':  [round(sil_full_pre,3), round(sil_full_tec,3)],
    'Davies-Bouldin': [round(db_full_pre,3), round(db_full_tec,3)],
    'Inercia K=4': [round(iner_pre,0), round(iner_tec,0)],
    'K elegido':   [K, K],
    'Features':    [len(feat_pre), len(feat_tec)],
})
print("=== MÉTRICAS GLOBALES ===")
print(global_metrics.to_string(index=False))
print()

# Tabla por cluster
rows = []
for k in range(K):
    mp = lbl_pre==k; mt = lbl_tec==k
    rows.append({
        'Grupo': 'Pre-TEC21', 'Cluster': f'C{k}',
        'n': int(mp.sum()), '% del grupo': round(mp.sum()/len(pre_df)*100,1),
        'Dropout (%)': round(y_pre[mp].mean()*100,2),
        'PNA media': round(pre_df['PNA'].values[mp].mean(),1),
        '% extranjero': round(pre_df['is_foreign'].values[mp].mean()*100,1),
        '% Prepa Tec': round(pre_df['estuvo.prepa_tec'].values[mp].mean()*100,1),
        '% Primera gen.': round(pre_df['first.generation.yes'].values[mp].mean()*100,1),
        '% con beca': round(pre_df['total.scholarship.loan'].values[mp].mean()*100,1),
    })
for k in range(K):
    mp = lbl_pre==k; mt = lbl_tec==k
    rows.append({
        'Grupo': 'TEC21', 'Cluster': f'C{k}',
        'n': int(mt.sum()), '% del grupo': round(mt.sum()/len(tec_df)*100,1),
        'Dropout (%)': round(y_tec[mt].mean()*100,2),
        'PNA media': round(tec_df['PNA'].values[mt].mean(),1),
        '% extranjero': round(tec_df['is_foreign'].values[mt].mean()*100,1),
        '% Prepa Tec': round(tec_df['estuvo.prepa_tec'].values[mt].mean()*100,1),
        '% Primera gen.': round(tec_df['first.generation.yes'].values[mt].mean()*100,1),
        '% con beca': round(tec_df['total.scholarship.loan'].values[mt].mean()*100,1),
    })

df_metrics = pd.DataFrame(rows)
print("=== MÉTRICAS POR CLUSTER ===")
print(df_metrics.to_string(index=False))
df_metrics.to_csv(OUT+'v4_metricas_clusters.csv', index=False)
print("\n✓ v4_metricas_clusters.csv guardado")


=== MÉTRICAS GLOBALES ===
    Grupo  n total  Dropout global (%)  Silhouette  Davies-Bouldin  Inercia K=4  K elegido  Features
Pre-TEC21    53010            8.837955       0.226           1.236     264531.0          4         8
    TEC21    24507            8.683233       0.226           1.499     122943.0          4         8

=== MÉTRICAS POR CLUSTER ===
    Grupo Cluster     n  % del grupo  Dropout (%)  PNA media  % extranjero  % Prepa Tec  % Primera gen.  % con beca
Pre-TEC21      C0 21262         40.1         5.77       91.6           0.0         47.5             4.9        51.7
Pre-TEC21      C1 29680         56.0        10.58       84.3           0.0         51.1             5.0         4.3
Pre-TEC21      C2  1667          3.1        14.22       88.0         100.0          0.4             4.4        13.0
Pre-TEC21      C3   401          0.8        20.20       86.8           7.0         43.6            12.2        13.7
    TEC21      C0  9248         37.7         4.73       92.2 

## B. Gráficas K-Means: convergencia y separación

### B.1 Convergencia del algoritmo (inercia por iteración)


In [106]:
# Inercia por número de iteraciones (convergencia)
def track_convergence(X, k=4, max_iter=40):
    """Rastrea la inercia iteración por iteración."""
    # Primera corrida para obtener centros iniciales
    km0 = KMeans(n_clusters=k, init='k-means++', n_init=1, max_iter=1, random_state=SEED)
    km0.fit(X)
    centers = km0.cluster_centers_.copy()
    inertias_iter = [km0.inertia_]
    for it in range(2, max_iter+1):
        km_it = KMeans(n_clusters=k, init=centers, n_init=1, max_iter=it, tol=1e-20, random_state=SEED)
        km_it.fit(X)
        inertias_iter.append(km_it.inertia_)
        centers = km_it.cluster_centers_.copy()
        if len(inertias_iter) > 1 and abs(inertias_iter[-2] - inertias_iter[-1]) < 1e-4:
            break
    return inertias_iter

conv_pre = track_convergence(X_pre)
conv_tec = track_convergence(X_tec)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('K-Means: Convergencia · Separación entre Clusters · Silhouette por K',
             fontsize=13, fontweight='bold')

# 1. Convergencia
ax = axes[0]
ax.plot(range(1, len(conv_pre)+1), conv_pre, 'o-', color='#3b82f6', lw=2, ms=5, label='Pre-TEC21')
ax.plot(range(1, len(conv_tec)+1), conv_tec, 's-', color='#f59e0b', lw=2, ms=5, label='TEC21')
ax.set_xlabel('Iteración'); ax.set_ylabel('Inercia (WCSS)')
ax.set_title('Convergencia K-Means (K=4)', fontweight='bold')
ax.legend(); ax.grid(alpha=.3)

# 2. Distancia inter-centroide (separación entre clusters)
ax2 = axes[1]
palette4 = ['#22c55e','#3b82f6','#f59e0b','#ef4444']
for group_name, X, labels in [('Pre-TEC21', X_pre, lbl_pre), ('TEC21', X_tec, lbl_tec)]:
    centroids = np.array([X[labels==k].mean(axis=0) for k in range(K)])
    dists = []
    pairs = []
    for i in range(K):
        for j in range(i+1, K):
            d = np.linalg.norm(centroids[i]-centroids[j])
            dists.append(d)
            pairs.append(f'C{i}-C{j}')
    x_pos = np.arange(len(pairs))
    offset = -0.2 if group_name == 'Pre-TEC21' else 0.2
    ax2.bar(x_pos+offset, dists, width=0.38,
            label=group_name,
            color='#3b82f6' if group_name=='Pre-TEC21' else '#f59e0b', alpha=.85)
ax2.set_xticks(range(len(pairs))); ax2.set_xticklabels(pairs, fontsize=9, rotation=20)
ax2.set_ylabel('Distancia euclidiana'); ax2.set_title('Distancia entre centroides', fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=.3)

# 3. Silhouette por K
ax3 = axes[2]
ax3.plot(list(K_RANGE), s_pre, 'o-', color='#3b82f6', lw=2, ms=6, label='Pre-TEC21')
ax3.plot(list(K_RANGE), s_tec, 's-', color='#f59e0b', lw=2, ms=6, label='TEC21')
ax3.axvline(K, color='red', ls='--', lw=1.5, label=f'K={K} elegido')
ax3.set_xlabel('K'); ax3.set_ylabel('Silhouette Score')
ax3.set_title('Silhouette por K', fontweight='bold')
ax3.legend(); ax3.grid(alpha=.3)

plt.tight_layout()
plt.show()


### B.2 Radar chart — perfil de cada cluster

In [107]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

def radar_chart(ax, values, labels, title, palette, n_clusters):
    N = len(labels)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([l.replace(' ', '\n') for l in labels], size=8)
    ax.set_ylim(-2.5, 2.5)
    ax.set_yticks([-2,-1,0,1,2]); ax.set_yticklabels(['-2','-1','0','1','2'], size=7)
    ax.grid(color='grey', linewidth=0.5, alpha=0.5)
    ax.set_title(title, size=11, fontweight='bold', pad=14)

    for k in range(n_clusters):
        vals = list(values[k]) + [values[k][0]]
        ax.plot(angles, vals, 'o-', lw=2, color=palette[k], label=f'C{k}')
        ax.fill(angles, vals, alpha=0.08, color=palette[k])
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)

# Extraer z-scores
feat_cols_pre = [c for c in prof_pre.columns if c not in ['Deserción (%)', 'n', 'grupo']]
feat_cols_tec = [c for c in prof_tec.columns if c not in ['Deserción (%)', 'n', 'grupo']]

vals_pre = prof_pre[feat_cols_pre].values.clip(-2.5, 2.5)
vals_tec = prof_tec[feat_cols_tec].values.clip(-2.5, 2.5)

pretty_pre = [LABELS.get(f, f) for f in feat_cols_pre]
pretty_tec = [LABELS.get(f, f) for f in feat_cols_tec]

fig = plt.figure(figsize=(16, 6))
fig.suptitle('Radar Chart de Perfiles por Cluster (z-scores)', fontsize=14, fontweight='bold')

ax1 = fig.add_subplot(121, polar=True)
ax2 = fig.add_subplot(122, polar=True)

radar_chart(ax1, vals_pre, pretty_pre, 'Pre-TEC21', palette, K)
radar_chart(ax2, vals_tec, pretty_tec, 'TEC21', palette, K)

plt.tight_layout()
plt.show()


## C. Variables más relevantes por cluster

Para cada cluster se calcula:
- **Peso discriminante** = |z-score de la media del cluster| → qué tan lejos está del promedio global
- Se muestran las variables que más diferencian a cada cluster del resto


In [108]:
def top_features_cluster(profile, n_top=4, group='Grupo'):
    feat_cols = [c for c in profile.columns if c not in ['Deserción (%)', 'n', 'grupo']]
    results = []
    for k in range(K):
        z_vals = profile.loc[k, feat_cols].astype(float)
        sorted_abs = z_vals.abs().sort_values(ascending=False)
        for feat in sorted_abs.index[:n_top]:
            z  = z_vals[feat]
            results.append({
                'Grupo': group, 'Cluster': f'C{k}',
                'Variable': LABELS.get(feat, feat),
                'z-score': round(z, 3),
                'Dirección': '▲ Alta' if z > 0 else '▼ Baja',
                '|z-score|': round(abs(z), 3),
            })
    return pd.DataFrame(results)

top_pre = top_features_cluster(prof_pre, n_top=4, group='Pre-TEC21')
top_tec = top_features_cluster(prof_tec, n_top=4, group='TEC21')
top_all = pd.concat([top_pre, top_tec])

print("=== VARIABLES MÁS DISCRIMINANTES POR CLUSTER ===\n")
for grupo, gdf in top_all.groupby('Grupo'):
    print(f"── {grupo} ──")
    for cluster, cdf in gdf.groupby('Cluster'):
        drop_rate = prof_pre.loc[int(cluster[1]),'Deserción (%)'] if grupo=='Pre-TEC21'                     else prof_tec.loc[int(cluster[1]),'Deserción (%)']
        print(f"  {cluster} (deserción={drop_rate:.1f}%):")
        for _, row in cdf.iterrows():
            print(f"    {row['Dirección']}  {row['Variable']:20s}  z={row['z-score']:+.3f}")
    print()

top_all.to_csv(OUT+'v4_top_features.csv', index=False)


=== VARIABLES MÁS DISCRIMINANTES POR CLUSTER ===

── Pre-TEC21 ──
  C0 (deserción=5.8%):
    ▲ Alta  Beca/Préstamo         z=+0.993
    ▲ Alta  PNA                   z=+0.681
    ▼ Baja  Extranjero            z=-0.182
    ▲ Alta  Extracurricular       z=+0.087
  C1 (deserción=10.6%):
    ▼ Baja  Beca/Préstamo         z=-0.686
    ▼ Baja  PNA                   z=-0.492
    ▼ Baja  Extranjero            z=-0.182
    ▲ Alta  Extracurricular       z=+0.087
  C2 (deserción=14.2%):
    ▲ Alta  Extranjero            z=+5.502
    ▼ Baja  Prepa Tec             z=-0.952
    ▼ Baja  Beca/Préstamo         z=-0.377
    ▲ Alta  Carga (FTE)           z=+0.099
  C3 (deserción=20.2%):
    ▼ Baja  Extracurricular       z=-11.454
    ▲ Alta  Padres EXATEC         z=+0.441
    ▼ Baja  Carga (FTE)           z=-0.438
    ▼ Baja  Beca/Préstamo         z=-0.353

── TEC21 ──
  C0 (deserción=4.7%):
    ▲ Alta  Beca/Préstamo         z=+0.933
    ▲ Alta  PNA                   z=+0.674
    ▼ Baja  Primera gen.    

In [109]:
fig, axes = plt.subplots(2, K, figsize=(20, 10))
fig.suptitle('Variables Más Discriminantes por Cluster (|z-score| vs promedio global)\n'
             'Pre-TEC21 (arriba) · TEC21 (abajo)', fontsize=13, fontweight='bold')

for row_i, (profile, group_name) in enumerate([(prof_pre,'Pre-TEC21'),(prof_tec,'TEC21')]):
    feat_cols = [c for c in profile.columns if c not in ['Deserción (%)', 'n', 'grupo']]
    for k in range(K):
        ax    = axes[row_i, k]
        z_vals = profile.loc[k, feat_cols].astype(float)
        sorted_idx = z_vals.abs().sort_values(ascending=False).index
        top_feats  = sorted_idx[:6]
        top_vals   = z_vals[top_feats].values
        labels_f   = [LABELS.get(f, f) for f in top_feats]
        colors_f   = ['#22c55e' if v>0 else '#ef4444' for v in top_vals]
        drop_k     = profile.loc[k,'Deserción (%)']
        n_k        = int(profile.loc[k,'n'])
        
        bars = ax.barh(range(len(top_feats)), top_vals, color=colors_f, edgecolor='white', lw=1)
        ax.set_yticks(range(len(top_feats)))
        ax.set_yticklabels(labels_f, fontsize=9)
        ax.axvline(0, color='black', lw=1)
        ax.set_xlim(-2.5, 2.5)
        ax.set_xlabel('z-score')
        ax.set_title(f'{group_name} — C{k}\n'
                     f'Deserción: {drop_k:.1f}%  |  n={n_k:,}',
                     fontsize=10, fontweight='bold',
                     color=palette[k])
        for bar, val in zip(bars, top_vals):
            ax.text(val + (0.08 if val >= 0 else -0.08), bar.get_y() + bar.get_height()/2,
                   f'{val:+.2f}', va='center',
                   ha='left' if val >= 0 else 'right', fontsize=8)
        ax.grid(axis='x', alpha=.3)
        # Indicador de riesgo
        risk_color = palette[k]
        ax.spines['top'].set_color(risk_color); ax.spines['top'].set_linewidth(3)
        ax.spines['right'].set_color(risk_color); ax.spines['right'].set_linewidth(3)

plt.tight_layout()
plt.show()


## D. Descripción e interpretación de clusters

### Convención de colores
- 🟢 **C0** — Menor riesgo (dropout más bajo)
- 🔵 **C1** — Riesgo medio-bajo
- 🟡 **C2** — Riesgo medio-alto
- 🔴 **C3** — Mayor riesgo

---

## D.1 Pre-TEC21

### 🟢 C0 — "Alta PNA + Beca activa" · n=21,262 (40.1%) · Deserción 5.8%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| PNA | **91.6** | +6.9 pts sobre C1 |
| Extracurriculares | **100%** | = C1, C2 |
| Beca/Préstamo | **51.7%** | muy por encima de C1 (4.3%) |
| Prepa Tec | 47.5% | mixto |
| Extranjero | 0% | nacional |
| Primera gen. | 4.9% | muy bajo |

**Perfil:** Estudiantes nacionales con el mejor desempeño académico de ingreso y cobertura financiera significativa (beca o préstamo). Activos en extracurriculares. La combinación de buen PNA y apoyo financiero los protege de la deserción. **Grupo con menor riesgo del sistema Pre-TEC21.**

**Variable más discriminante:** `total.scholarship.loan` (z=+2.47) y `PNA` (z=+0.68)

---

### 🔵 C1 — "PNA bajo + Sin beca" · n=29,680 (56.0%) · Deserción 10.6%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| PNA | **84.3** | el más bajo entre nacionales |
| Beca/Préstamo | **4.3%** | muy por debajo de C0 |
| Extracurriculares | 100% | = C0 |
| Prepa Tec | 51.1% | mixto |
| Extranjero | 0% | nacional |

**Perfil:** Estudiantes nacionales sin apoyo financiero y PNA más bajo. Es el **cluster mayoritario** (más de la mitad de Pre-TEC21). La ausencia de beca, combinada con menor desempeño en admisión, duplica el riesgo de deserción respecto a C0. Posiblemente el grupo con mayor necesidad de intervención por su tamaño.

**Variable más discriminante:** `total.scholarship.loan` (z=−1.71) y `PNA` (z=−0.49)

---

### 🟡 C2 — "Extranjeros sin Prepa Tec" · n=1,667 (3.1%) · Deserción 14.2%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| Extranjero | **100%** | exclusivo de este cluster |
| Prepa Tec | **0.4%** | prácticamente ninguno |
| PNA | 88.0 | nivel medio |
| Primera gen. | 4.4% | bajo |
| Beca | 13% | baja |

**Perfil:** Todos los estudiantes extranjeros se concentran aquí. PNA medio, sin Prepa Tec (vienen de sistemas educativos externos), bajo apoyo financiero. El riesgo elevado puede deberse a barreras de adaptación cultural, lejanía familiar y falta de cobertura económica.

**Variable más discriminante:** `is_foreign` (z=+8.33) — variable absolutamente dominante

---

### 🔴 C3 — "Sin extracurriculares" · n=401 (0.8%) · Deserción 20.2%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| Extracurriculares | **0%** | el único cluster sin actividades |
| Padres EXATEC | **25.4%** | mayor que cualquier otro cluster |
| PNA | 86.8 | nivel medio |
| Primera gen. | 12.2% | el más alto entre nacionales |
| Beca | 13.7% | baja |

**Perfil:** Grupo pequeño definido por la **ausencia total de actividades extracurriculares**. Paradójicamente tiene más padres EXATEC, lo que sugiere que la baja participación extracurricular es una elección o restricción particular. El mayor % de primera generación y la falta de involucramiento social explican la **mayor tasa de deserción de Pre-TEC21 (20.2%).**

**Variable más discriminante:** `has_extracurriculars` (z=−4.86) — absolutamente diferenciador

---

## D.2 TEC21

### 🟢 C0 — "Alta PNA + Beca + Extracurricular" · n=9,248 (37.7%) · Deserción 4.7%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| PNA | **92.2** | el más alto de TEC21 |
| Beca/Préstamo | **49.5%** | muy por encima de C1 |
| Extracurriculares | **88.3%** | alto |
| Padres EXATEC | 20.9% | más alto de nacionales |
| Primera gen. | 0% | ninguno |
| Extranjero | 0% | nacional |

**Perfil:** Espejo de Pre-C0. Mejor perfil académico y financiero de TEC21. Cero primera generación sugiere fuerte capital familiar educativo. La combinación de buen PNA, beca y extracurriculares produce el **dropout más bajo de todo el análisis (4.7%).**

---

### 🔵 C1 — "PNA bajo + Sin beca" · n=11,293 (46.1%) · Deserción 10.7%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| PNA | **84.6** | el más bajo entre nacionales TEC21 |
| Beca/Préstamo | **3.3%** | casi ninguna |
| Prepa Tec | 55.3% | la mayoría con Prepa Tec |
| Primera gen. | 0% | ninguno |
| Extracurriculares | 72.8% | moderado |

**Perfil:** Similar a Pre-C1. Bajo PNA, sin beca. La mayor proporción con Prepa Tec no los protege sin apoyo financiero. El grupo más numeroso de TEC21 con riesgo moderado-alto.

---

### 🟡 C2 — "Extranjeros" · n=943 (3.8%) · Deserción 10.8%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| Extranjero | **100%** | exclusivo de este cluster |
| Prepa Tec | **0.1%** | casi ninguno |
| Primera gen. | 11.1% | algo mayor |
| Beca | 14.3% | baja |
| PNA | 88.8 | nivel medio |

**Perfil:** Equivalente exacto de Pre-C2. Todos los extranjeros de TEC21. La similitud coseno con Pre-C2 es **0.999** — los extranjeros forman el mismo perfil en ambos modelos educativos. Destaca que en TEC21 su deserción (10.8%) es menor que en Pre-TEC21 (14.2%), posiblemente por mejor soporte institucional del modelo nuevo.

---

### 🔴 C3 — "Primera generación universitaria" · n=3,023 (12.3%) · Deserción 12.8%

| Variable | Valor real | vs grupo |
|----------|-----------|---------|
| Primera gen. | **100%** | exclusivo de este cluster |
| Padres EXATEC | **1.4%** | prácticamente ninguno |
| Extracurriculares | 82.4% | activos |
| Beca | 22.0% | moderada |
| PNA | 88.2 | nivel medio |
| Extranjero | 0% | nacional |

**Perfil:** Todos los estudiantes de primera generación de TEC21 se agrupan aquí. Sin padres EXATEC, sin capital institucional Tec previo. Aunque tienen PNA razonable y participan en extracurriculares, el hecho de ser los primeros en su familia en estudiar universitario implica menor red de apoyo y orientación. Este perfil no existe de forma tan compacta en Pre-TEC21, donde la primera generación se disuelve entre otros clusters.


## E. Tabla comparativa del matching Pre-TEC21 ↔ TEC21

### Interpretación por par


In [110]:
# Tabla resumen del matching
match_summary = pd.read_csv(OUT+'v4_matching.csv')
print(match_summary.to_string(index=False))
print()

# Análisis de diferencias de dropout por par
print("=== DIFERENCIA DE DROPOUT ENTRE MODELOS EDUCATIVOS ===")
for _, row in match_summary.iterrows():
    diff = row['dropout_tec_%'] - row['dropout_pre_%']
    trend = '↑ TEC21 más alto' if diff > 0.5 else ('↓ TEC21 más bajo' if diff < -0.5 else '≈ similar')
    print(f"  {row['par']:25s}  Pre={row['dropout_pre_%']:.1f}%  TEC={row['dropout_tec_%']:.1f}%  Δ={diff:+.1f}pp  {trend}")


          par  pre_cluster  tec_cluster  similitud  dropout_pre_%  dropout_tec_%  n_pre  n_tec calidad
Pre-C0↔TEC-C0            0            0     0.9458           5.77           4.73  21262   9248  sólido
Pre-C1↔TEC-C1            1            1     0.8844          10.58          10.65  29680  11293  sólido
Pre-C2↔TEC-C2            2            2     0.9986          14.22          10.82   1667    943  sólido
Pre-C3↔TEC-C3            3            3     0.0000          20.20          12.77    401   3023   débil

=== DIFERENCIA DE DROPOUT ENTRE MODELOS EDUCATIVOS ===
  Pre-C0↔TEC-C0              Pre=5.8%  TEC=4.7%  Δ=-1.0pp  ↓ TEC21 más bajo
  Pre-C1↔TEC-C1              Pre=10.6%  TEC=10.7%  Δ=+0.1pp  ≈ similar
  Pre-C2↔TEC-C2              Pre=14.2%  TEC=10.8%  Δ=-3.4pp  ↓ TEC21 más bajo
  Pre-C3↔TEC-C3              Pre=20.2%  TEC=12.8%  Δ=-7.4pp  ↓ TEC21 más bajo


## F. Gráfica comparativa final — perfiles y matching

In [111]:
fig = plt.figure(figsize=(22, 14))
fig.suptitle('Resumen Visual: Clusters Pre-TEC21 vs TEC21\n'
             'Perfiles · Deserción · Matching · Variables Clave',
             fontsize=15, fontweight='bold', y=1.01)

# ── Layout manual ─────────────────────────────────────────────────────────────
gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.4)

# Panel 1: Deserción por cluster comparado
ax1 = fig.add_subplot(gs[0, :2])
x, w = np.arange(K), 0.35
b1 = ax1.bar(x-w/2, prof_pre['Deserción (%)'], w, label='Pre-TEC21', color='#3b82f6', alpha=.85)
b2 = ax1.bar(x+w/2, prof_tec['Deserción (%)'], w, label='TEC21',     color='#f59e0b', alpha=.85)
ax1.axhline(y_pre.mean()*100, color='#3b82f6', ls='--', lw=1.3, alpha=.6)
ax1.axhline(y_tec.mean()*100, color='#f59e0b', ls='--', lw=1.3, alpha=.6)
ax1.set_xticks(x); ax1.set_xticklabels([f'C{k}' for k in range(K)], fontsize=11)
ax1.set_ylabel('Deserción (%)'); ax1.set_title('Tasa de Deserción por Cluster', fontweight='bold')
ax1.legend()
for bars in [b1, b2]:
    for bar in bars:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.2,
                f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Panel 2: Tamaño de grupos (pie)
ax2 = fig.add_subplot(gs[0, 2])
sizes_pre = [int((lbl_pre==k).sum()) for k in range(K)]
wedges, texts, autotexts = ax2.pie(sizes_pre, colors=palette, autopct='%1.1f%%',
                                    startangle=90, pctdistance=0.7)
for at in autotexts: at.set_fontsize(9); at.set_fontweight('bold')
ax2.set_title('Pre-TEC21\nDistribución clusters', fontweight='bold', fontsize=10)
ax2.legend([f'C{k}' for k in range(K)], loc='lower right', fontsize=8)

ax3 = fig.add_subplot(gs[0, 3])
sizes_tec = [int((lbl_tec==k).sum()) for k in range(K)]
wedges, texts, autotexts = ax3.pie(sizes_tec, colors=palette, autopct='%1.1f%%',
                                    startangle=90, pctdistance=0.7)
for at in autotexts: at.set_fontsize(9); at.set_fontweight('bold')
ax3.set_title('TEC21\nDistribución clusters', fontweight='bold', fontsize=10)
ax3.legend([f'C{k}' for k in range(K)], loc='lower right', fontsize=8)

# Panel 3: Heatmap compacto Pre-TEC21
ax4 = fig.add_subplot(gs[1, :2])
feat_cols_p = [c for c in prof_pre.columns if c not in ['Deserción (%)', 'n', 'grupo']]
data_p = prof_pre[feat_cols_p].values.astype(float)
im4 = ax4.imshow(data_p, cmap='RdYlGn_r', aspect='auto', vmin=-2.5, vmax=2.5)
ax4.set_xticks(range(len(feat_cols_p)))
ax4.set_xticklabels([LABELS.get(f,f) for f in feat_cols_p], rotation=35, ha='right', fontsize=8)
ax4.set_yticks(range(K))
labels_pre_k = [
    f'C0 · {y_pre[lbl_pre==0].mean()*100:.1f}%',
    f'C1 · {y_pre[lbl_pre==1].mean()*100:.1f}%',
    f'C2 · {y_pre[lbl_pre==2].mean()*100:.1f}%',
    f'C3 · {y_pre[lbl_pre==3].mean()*100:.1f}%',
]
ax4.set_yticklabels(labels_pre_k, fontsize=9, fontweight='bold')
ax4.set_title('Pre-TEC21 — z-scores (dropout por cluster)', fontweight='bold', fontsize=10)
plt.colorbar(im4, ax=ax4, fraction=0.025, pad=0.01)
for i in range(K):
    for j in range(len(feat_cols_p)):
        v = data_p[i,j]
        ax4.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=7,
                color='white' if abs(v)>1.5 else 'black')

# Panel 4: Heatmap compacto TEC21
ax5 = fig.add_subplot(gs[1, 2:])
feat_cols_t = [c for c in prof_tec.columns if c not in ['Deserción (%)', 'n', 'grupo']]
data_t = prof_tec[feat_cols_t].values.astype(float)
im5 = ax5.imshow(data_t, cmap='RdYlGn_r', aspect='auto', vmin=-2.5, vmax=2.5)
ax5.set_xticks(range(len(feat_cols_t)))
ax5.set_xticklabels([LABELS.get(f,f) for f in feat_cols_t], rotation=35, ha='right', fontsize=8)
ax5.set_yticks(range(K))
labels_tec_k = [
    f'C0 · {y_tec[lbl_tec==0].mean()*100:.1f}%',
    f'C1 · {y_tec[lbl_tec==1].mean()*100:.1f}%',
    f'C2 · {y_tec[lbl_tec==2].mean()*100:.1f}%',
    f'C3 · {y_tec[lbl_tec==3].mean()*100:.1f}%',
]
ax5.set_yticklabels(labels_tec_k, fontsize=9, fontweight='bold')
ax5.set_title('TEC21 — z-scores (dropout por cluster)', fontweight='bold', fontsize=10)
plt.colorbar(im5, ax=ax5, fraction=0.025, pad=0.01)
for i in range(K):
    for j in range(len(feat_cols_t)):
        v = data_t[i,j]
        ax5.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=7,
                color='white' if abs(v)>1.5 else 'black')

# Panel 5: Matching visual
ax6 = fig.add_subplot(gs[2, :])
ax6.set_xlim(0, 22); ax6.set_ylim(-0.6, K-0.4); ax6.axis('off')
ax6.set_title('Matching Óptimo Pre-TEC21 ↔ TEC21 (Algoritmo Húngaro · Similitud Coseno)',
              fontweight='bold', fontsize=11)

cluster_desc_pre = [
    'C0 · Alta PNA + Beca',
    'C1 · PNA bajo + Sin beca',
    'C2 · Extranjeros',
    'C3 · Sin extracurriculares',
]
cluster_desc_tec = [
    'C0 · Alta PNA + Beca',
    'C1 · PNA bajo + Sin beca',
    'C2 · Extranjeros',
    'C3 · Primera generación',
]

yp = list(range(K-1,-1,-1))
for k in range(K):
    col = palette[k]
    n_p  = int((lbl_pre==k).sum()); dr_p = y_pre[lbl_pre==k].mean()*100
    n_t  = int((lbl_tec==k).sum()); dr_t = y_tec[lbl_tec==k].mean()*100
    # Pre
    ax6.add_patch(mpatches.FancyBboxPatch((.1,yp[k]-.32),6.5,.64,
        boxstyle='round,pad=0.05',facecolor=col+'22',edgecolor=col,lw=2))
    ax6.text(3.35,yp[k]+.1,cluster_desc_pre[k],fontsize=10,fontweight='bold',color=col,ha='center')
    ax6.text(3.35,yp[k]-.18,f'n={n_p:,} · dropout={dr_p:.1f}%',fontsize=8,color='#444',ha='center')
    # TEC
    ax6.add_patch(mpatches.FancyBboxPatch((15.4,yp[k]-.32),6.5,.64,
        boxstyle='round,pad=0.05',facecolor=col+'22',edgecolor=col,lw=2))
    ax6.text(18.65,yp[k]+.1,cluster_desc_tec[k],fontsize=10,fontweight='bold',color=col,ha='center')
    ax6.text(18.65,yp[k]-.18,f'n={n_t:,} · dropout={dr_t:.1f}%',fontsize=8,color='#444',ha='center')

ax6.text(11,K-.3,'Pre-TEC21',fontsize=13,fontweight='bold',color='#3b82f6',ha='center')
ax6.text(11,-.45,'TEC21',fontsize=13,fontweight='bold',color='#f59e0b',ha='center')

for r,c,s in match_pairs:
    col  = '#15803d' if s>=.6 else ('#d97706' if s>=.15 else '#b91c1c')
    lw   = 1 + 5*min(s, 1.0)
    ax6.annotate('',xy=(15.3,yp[c]),xytext=(6.7,yp[r]),
        arrowprops=dict(arrowstyle='->',color=col,lw=lw))
    mid_y = (yp[r]+yp[c])/2
    quality = '🟢' if s>=.6 else ('🟡' if s>=.15 else '🔴')
    ax6.text(11,mid_y,f'{quality} sim={s:.3f}',
        fontsize=9.5,ha='center',color=col,fontweight='bold',
        bbox=dict(facecolor='white',edgecolor=col,pad=2.5,alpha=.9,
                  boxstyle='round,pad=0.3'))

plt.show()
